In [ ]:
import os
from pathlib import Path

import numpy as np
from torchvision import transforms
from medmnist import PathMNIST

# turn off GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((32, 32)),
        transforms.Normalize(mean=[.5], std=[.5]) 
    ])

dataset_orig = PathMNIST(split="test", download=True, size=128)
dataset_trans = PathMNIST(split="test", download=True, size=64, transform=transform)

In [ ]:
dataset_orig

In [ ]:
os.environ['PROJECT_ROOT'] = '/home/.../slm_boosting/PEXEL'
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"])
%cd $PROJECT_ROOT

In [ ]:
labels = ['adipose', 'background', 'debris', 'lymphocytes', 'mucus', 'smooth muscle', 'normal colon mucosa', 'cancer-associated stroma', 'colorectal adenocarcinoma epithelium']

In [ ]:
import matplotlib.pyplot as plt

idx = 48
img = dataset_orig[idx][0]
label = dataset_orig[idx][1][0]
# save image
img.save('plots/histo_image.png')

display(img)
print("Correct class: " + labels[label])

In [ ]:
# load models
import torch

from project_files.utils import load_json
from project_files.vision.pathMNIST.models.classifiers import create_pathMNIST_classifier

cfg = load_json(os.path.join("project_files/vision/pathMNIST/configs", "train_ce_dummy_adam.json"))
ce_model = create_pathMNIST_classifier(
    constraint_handler="dummy", 
    loss_func="ce", 
    model_params=cfg["model_params"], 
    constraint_params=cfg["constraint_params"]
)
# load best checkpoint
ce_model.load_state_dict(
    torch.load(os.path.join("project_files/vision/pathMNIST/checkpoints/ce_smaller_set", "last.ckpt"), map_location="cpu")["state_dict"]
    )

cfg = load_json(os.path.join("project_files/vision/pathMNIST/configs", "train_exp-loss_logsumexp-penalty_adam_smaller_set.json"))
penex_model = create_pathMNIST_classifier(
    constraint_handler="logsumexp-penalty", 
    loss_func="exp-loss", 
    model_params=cfg["model_params"], 
    constraint_params=cfg["constraint_params"]
)
# load best checkpoint
penex_model.load_state_dict(
    torch.load(os.path.join("project_files/vision/pathMNIST/checkpoints/penex_smaller_set", "last.ckpt"), map_location="cpu")["state_dict"]
)

cfg = load_json(os.path.join("project_files/vision/pathMNIST/configs", "train_ce_dummy_adam_smoothing_smaller_set.json"))
smoothing_model = create_pathMNIST_classifier(
    constraint_handler="dummy", 
    loss_func="ce", 
    model_params=cfg["model_params"], 
    constraint_params=cfg["constraint_params"]
)
# load best checkpoint
smoothing_model.load_state_dict(
    torch.load(os.path.join("project_files/vision/pathMNIST/checkpoints/smoothing_smaller_set", "last.ckpt"), map_location="cpu")["state_dict"]
    )

In [ ]:
models = [ce_model, smoothing_model, penex_model]
model_names = ["cross-entropy", "label smoothing", "PENEX"]
model_color_nrs = [0, 2, 4]
width = 0.25
offsets = [-width, 0, width]

In [ ]:
plt.style.use(["science", "vibrant"])
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern Roman']
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.fancybox'] = True

color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
import textwrap

probs = {}
for i, model in enumerate(models):
    model.eval()
    with torch.no_grad():
        logits = model(dataset_trans[idx][0].unsqueeze(0))
        probs_ = torch.softmax(logits, dim=-1)[0]         # shape: (num_classes,)

    probs[model_names[i]] = probs_.cpu().numpy()

class_names = [textwrap.fill(labels[i], width=20) for i in range(len(probs_))]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FixedLocator, FixedFormatter

# … (compute your `probs` dict and `class_names` as before) …

plt.style.use(["science", "vibrant"])
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['Computer Modern Roman']
plt.rcParams['legend.frameon']  = True
plt.rcParams['legend.fancybox'] = True

fig, ax = plt.subplots(figsize=(6, 3))
x = np.arange(len(probs_))
width = 0.2
offsets = np.linspace(-width, width, len(models))

for i, name in enumerate(model_names):
    ax.bar(x + offsets[i], probs[name], width, label=name, color=color_cycle[model_color_nrs[i]])

# Now override the ticks: only every 5th class
step = 1
locs = x[::step]
labels = [class_names[j] for j in locs]
ax.xaxis.set_major_locator(FixedLocator(locs))
ax.xaxis.set_major_formatter(FixedFormatter(labels))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)


ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout()
plt.savefig("plots/histo_predictions.pdf", bbox_inches='tight')
plt.show()